# Source Attestation & Trust Calibration — Analysis Notebook

Analysis pipeline for the 3×2×2 within-subjects vignette study (attestation × correctness × stakes).

**What this notebook does**
1. Loads a coded long-format export and validates the schema/design balance.
2. Descriptives for the three outcomes (trust, reliance, perceived consequence) by condition.
3. Crossed random-effects mixed models (participant + stem) for each outcome — **H1, H2, H4**.
4. An H4 interaction model (attestation × correctness) and a stakes-moderation model.
5. A scaffold for **H3** (mediation: attestation/correctness → trust → reliance) — *runs a basic path version; the full bootstrapped mediation is left for you to specify.*
6. Exports a tidy results table.

**What this notebook does NOT do:** interpret results, write prose, or decide which model is 'the' headline. It produces numbers; the reading of them is yours.

Expected columns: `response_id, version, trial_position, stem_id, stakes, att_level, correctness, trust_num, reliance_code, consequence_num, att_code, corr_code, stakes_code`.

Coding convention (from the prep pipeline): `att_code` ∈ {0,1,2} (none/weak/strong, ordinal); `corr_code`, `stakes_code` ∈ {−0.5,+0.5} (contrast-coded).

## 0. Setup

In [3]:
# If needed:
# %pip install pandas numpy statsmodels scipy matplotlib

import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')
pd.set_option('display.max_columns', None)

# ---- EDIT THIS PATH ----
DATA_PATH = 'long_coded.csv'
# ------------------------

ATT_ORDER = ['none', 'weak', 'strong']

## 1. Load & validate

In [4]:
df = pd.read_csv(DATA_PATH)

required = ['response_id', 'stem_id', 'stakes', 'att_level', 'correctness',
            'trust_num', 'reliance_code', 'consequence_num',
            'att_code', 'corr_code', 'stakes_code']
missing = [c for c in required if c not in df.columns]
assert not missing, f'Missing required columns: {missing}'

n_part = df['response_id'].nunique()
trials = df.groupby('response_id').size()
print(f'Participants: {n_part}')
print(f'Rows (participant-trials): {len(df)}')
print(f'Trials per participant — min {trials.min()}, median {int(trials.median())}, max {trials.max()}')
print(f'Unique stems: {df.stem_id.nunique()}')
if "version" in df.columns:
    print(f'Versions present: {sorted(df.version.unique())}')

# Coding sanity checks
print('\nCoding check:')
print('  att_code   :', sorted(df.att_code.dropna().unique()), '(expect 0,1,2)')
print('  corr_code  :', sorted(df.corr_code.dropna().unique()), '(expect -0.5,0.5)')
print('  stakes_code:', sorted(df.stakes_code.dropna().unique()), '(expect -0.5,0.5)')
print('  trust_num  :', df.trust_num.min(), 'to', df.trust_num.max(), '(expect 1-7)')
print('  reliance   :', sorted(df.reliance_code.dropna().unique()), '(expect 1,2,3)')
print('  consequence:', df.consequence_num.min(), 'to', df.consequence_num.max(), '(expect 1-5)')

FileNotFoundError: [Errno 2] No such file or directory: 'long_coded.csv'

In [ ]:
# Design balance: cell counts for the full 3x2x2 crossing
cell_counts = (df.groupby(['att_level', 'correctness', 'stakes'])
                 .size().rename('n').reset_index())
cell_pivot = cell_counts.pivot_table(index=['att_level'],
                                     columns=['correctness', 'stakes'],
                                     values='n').reindex(ATT_ORDER)
print('Cell counts (rows = attestation):')
display(cell_pivot)

# Missingness on outcomes
print('\nMissing values on outcomes:')
print(df[['trust_num', 'reliance_code', 'consequence_num']].isna().sum())

## 2. Descriptives by condition

In [ ]:
def desc_by(col, by):
    g = df.groupby(by)[col].agg(['mean', 'std', 'count'])
    if by == 'att_level':
        g = g.reindex(ATT_ORDER)
    return g

print('=== TRUST (1-7) ===')
print('\nby attestation:'); display(desc_by('trust_num', 'att_level'))
print('by correctness:'); display(desc_by('trust_num', 'correctness'))
print('by stakes:'); display(desc_by('trust_num', 'stakes'))

print('\nTrust: attestation x correctness (means):')
display(df.pivot_table('trust_num', 'att_level', 'correctness', 'mean').reindex(ATT_ORDER))

In [ ]:
print('=== RELIANCE (1=Reject, 2=Verify, 3=Use) ===')
print('by attestation:'); display(desc_by('reliance_code', 'att_level'))
print('by correctness:'); display(desc_by('reliance_code', 'correctness'))
print('by stakes:'); display(desc_by('reliance_code', 'stakes'))

print('\n=== PERCEIVED CONSEQUENCE (1-5) ===')
print('by stakes (manipulation check):'); display(desc_by('consequence_num', 'stakes'))
print('by attestation:'); display(desc_by('consequence_num', 'att_level'))

## 3. Mixed models — crossed random effects (participant + stem)

Random intercepts for both participant (`response_id`) and item (`stem_id`), via a participant grouping with a variance component for stem. Fixed effects use the contrast/ordinal coding already in the data.

- **H1**: positive `att_code` slope on trust and reliance.
- **H2**: `att_code` effect persists controlling for `corr_code`; small/null `corr_code` effect = trust tracks the label, not truth.
- **H4**: `stakes_code` effect, and the attestation×stakes / correctness×stakes interactions.

In [ ]:
def fit_crossed(dv, data, extra_fixed=''):
    """Crossed RE: random intercept for participant + variance component for stem."""
    formula = f'{dv} ~ att_code + corr_code + stakes_code' + extra_fixed
    md = smf.mixedlm(formula, data, groups=data['response_id'],
                     re_formula='~1',
                     vc_formula={'stem': '0 + C(stem_id)'})
    return md.fit(reml=False)

print('============ TRUST — main-effects model ============')
m_trust = fit_crossed('trust_num', df)
print(m_trust.summary())

In [ ]:
print('============ RELIANCE — main-effects model ============')
print('(Reliance is ordinal 1-3; this LMM treats it as continuous for a fast read.')
print(' See Section 3b for the ordinal model.)')
m_rel = fit_crossed('reliance_code', df)
print(m_rel.summary())

print('\n============ PERCEIVED CONSEQUENCE — manipulation check ============')
m_cons = fit_crossed('consequence_num', df)
print(m_cons.summary())

### 3a. H4 — interaction & stakes moderation

In [ ]:
print('=== Trust: attestation x correctness (H4 core interaction) ===')
m_int = fit_crossed('trust_num', df, extra_fixed=' + att_code:corr_code')
print(m_int.summary().tables[1])

print('\n=== Trust: full stakes moderation (att x corr x stakes) ===')
m_full = smf.mixedlm('trust_num ~ att_code * corr_code * stakes_code',
                     df, groups=df['response_id'], re_formula='~1',
                     vc_formula={'stem': '0 + C(stem_id)'}).fit(reml=False)
print(m_full.summary().tables[1])

### 3b. Reliance as ordinal (robustness)

Reliance is a 3-level ordinal outcome. A proper random-effects ordinal model (e.g. a Bayesian/`ordinal`-package fit in R, or a GEE/clmm) is the principled choice. Here is a population-averaged ordinal check via `OrderedModel` (no random effects) and a GEE with exchangeable correlation by participant, as a sanity comparison to the LMM above.

In [ ]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

ord_df = df.dropna(subset=['reliance_code', 'att_code', 'corr_code', 'stakes_code']).copy()
ord_df['reliance_ord'] = ord_df['reliance_code'].astype(int)

om = OrderedModel(ord_df['reliance_ord'],
                  ord_df[['att_code', 'corr_code', 'stakes_code']],
                  distr='logit')
om_res = om.fit(method='bfgs', disp=False)
print('=== Ordinal logit (no RE) — reliance ===')
print(om_res.summary())

# GEE clustered by participant (population-averaged, accounts for within-person dependence)
print('\n=== GEE (clustered by participant) — reliance ===')
gee = smf.gee('reliance_code ~ att_code + corr_code + stakes_code',
              groups='response_id', data=ord_df,
              cov_struct=sm.cov_struct.Exchangeable()).fit()
print(gee.summary().tables[1])

## 4. H3 — mediation scaffold (attestation/correctness → trust → reliance)

**Not a finished mediation analysis.** This fits the two component paths so you can see the pieces; the full indirect-effect test (bootstrapped CI, or a multilevel mediation that respects the crossed structure) is left for you to specify and run. Naïvely treating multilevel data with single-level mediation will misestimate SEs — decide the approach before reporting.

Path a: predictors → trust (the `m_trust` model above).  
Path b + c': trust + predictors → reliance.

In [ ]:
# Path a: effect of att/corr on the mediator (trust) — already in m_trust
print('PATH a (predictors -> trust):')
print(m_trust.summary().tables[1].loc[['att_code', 'corr_code']])

# Path b & c': mediator + predictors -> reliance
print('\nPATH b/c\' (trust + predictors -> reliance):')
m_med = smf.mixedlm('reliance_code ~ trust_num + att_code + corr_code + stakes_code',
                    df, groups=df['response_id'], re_formula='~1',
                    vc_formula={'stem': '0 + C(stem_id)'}).fit(reml=False)
print(m_med.summary().tables[1])

print('\n# To complete H3: compute indirect effect (a*b) with a bootstrap CI,')
print('# or use a multilevel mediation framework. This cell only exposes the paths.')

## 5. Tidy results export

In [ ]:
def tidy(res, label):
    """Pull the coefficient table from a fitted mixedlm result into a tidy frame."""
    p = pd.DataFrame({
        'term': res.params.index,
        'coef': res.params.values,
        'std_err': res.bse.reindex(res.params.index).values,
        'z': res.tvalues.reindex(res.params.index).values,
        'p_value': res.pvalues.reindex(res.params.index).values,
    })
    p.insert(0, 'model', label)
    return p

results = pd.concat([
    tidy(m_trust, 'trust_main'),
    tidy(m_rel,   'reliance_main_LMM'),
    tidy(m_cons,  'consequence_main'),
    tidy(m_int,   'trust_attXcorr'),
], ignore_index=True)

display(results)
results.to_csv('analysis_results.csv', index=False)
print('Wrote analysis_results.csv')

## 6. Quick diagnostic plots (optional)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Trust by attestation x correctness
piv = df.pivot_table('trust_num', 'att_level', 'correctness', 'mean').reindex(ATT_ORDER)
piv.plot(marker='o', ax=axes[0])
axes[0].set_title('Trust by attestation x correctness')
axes[0].set_xlabel('attestation'); axes[0].set_ylabel('mean trust (1-7)')
axes[0].set_ylim(1, 7)

# Reliance by attestation x stakes
piv2 = df.pivot_table('reliance_code', 'att_level', 'stakes', 'mean').reindex(ATT_ORDER)
piv2.plot(marker='o', ax=axes[1])
axes[1].set_title('Reliance by attestation x stakes')
axes[1].set_xlabel('attestation'); axes[1].set_ylabel('mean reliance (1-3)')

plt.tight_layout()
plt.savefig('diagnostic_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Wrote diagnostic_plots.png')